# SpottingSalmon
YOLO model logging script
- To register the final model as an MLFlow model


In [0]:
%pip install --quiet numpy==1.26.4  mlflow ultralytics

In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow
import mlflow.pyfunc
import ultralytics
from mlflow.models.signature import ModelSignature
from mlflow.types import Schema, TensorSpec, DataType, ColSpec
import numpy as np
from ultralytics import YOLO
from pyspark.sql import functions as F

In [0]:
{
  "dataframe_split": {
    "columns": [
      "fish"
    ],
    "data": [
      [
        "/Volumes/prd_dash_lab/dash_data_science_unrestricted/shared_external_volume/rachels_stuff/4_2021-07-06_14-11-37x.mp4"
      ]
    ]
  }
}

### 1. Model logging

In [0]:
# Run ID and artifact relative path
run_id = "cc1ca4e121884869ac75efcba1a0af4a"
artifact_path = "weights/best.pt"

#model_path = f"runs:/{run_id}/{artifact_path}"
# Download to local temp dir
model_path = mlflow.artifacts.download_artifacts(
    run_id=run_id,
    artifact_path=artifact_path
)

print(f"Model downloaded to: {model_path}")

# Load the model
model = YOLO(model_path)

In [0]:
video_path = "/Volumes/prd_dash_lab/dash_data_science_unrestricted/shared_external_volume/rachels_stuff/4_2021-07-06_14-11-37x.mp4"
print("Video exists?", os.path.exists(video_path))

In [0]:
import cv2

cap = cv2.VideoCapture(video_path)
print("Opened?", cap.isOpened())

success, frame = cap.read()
print("First frame read:", success)
cap.release()

In [0]:
import os

video_path = "/Volumes/prd_dash_lab/dash_data_science_unrestricted/shared_external_volume/rachels_stuff/4_2021-07-06_14-11-37x.mp4"  # <-- update if needed
print("Video exists?", os.path.exists(video_path))

In [0]:
import mlflow.pyfunc
import pandas as pd
from ultralytics import YOLO
import cv2
import os
import sys

# The original version did not check whether the video path actually existed
# or whether OpenCV could open it. If the path was invalid (e.g. Model Serving
# not having access to /Volumes/...), VideoCapture returned no frames and the
# model returned an empty result. Added explicit checks.
def _log(msg):
    print(msg, flush=True)          # stdout (flushed)
    print(msg, file=sys.stderr, flush=True)  # stderr (flushed)

def read_video_frames(video_path):
    """Generator that yields frames from a video file as numpy arrays."""

    _log(f"[INFO] Attempting to read video: {video_path}")

    if not os.path.exists(video_path):
        raise FileNotFoundError(f"Video path does not exist or cannot be accessed: {video_path}")

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise RuntimeError(f"OpenCV could not open video file: {video_path}")

    _log(f"[INFO] Video opened successfully: {video_path}")

    frame_number = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            _log(f"[INFO] No more frames returned at frame {frame_number}")
            break

        # Print every 10 frames for benchmarking
        if frame_number % 10 == 0:
            _log(f"[INFO] Reading frame {frame_number}")

        yield frame_number, frame
        frame_number += 1

    cap.release()
    _log(f"[INFO] Finished reading all frames from: {video_path}")


# The original code assumed "results" was a LIST of Results objects.
# But YOLO(frame) returns a SINGLE Results object per frame, not a list.
# Ultralytics can sometimes return [Results] instead of Results.
# This caused: 'list' object has no attribute 'boxes'.
# The fix is to handle both cases properly.
def convert_yolo_to_dets(result):
    """Convert a single YOLO Results object to a list of detection dicts."""

    detections = []

    if result is None or result.boxes is None or len(result.boxes) == 0:
        _log("[INFO] YOLO detected 0 objects on this frame")
        return detections

    boxes = result.boxes
    xyxy = boxes.xyxy.cpu().numpy()
    confs = boxes.conf.cpu().numpy()

    _log(f"[INFO] YOLO detected {len(boxes)} object(s)")

    for (x1, y1, x2, y2), conf in zip(xyxy, confs):
        detections.append({
            "x1": float(x1),
            "y1": float(y1),
            "x2": float(x2),
            "y2": float(y2),
            "confidence": float(conf)
        })

    return detections


class FishVideoDetector(mlflow.pyfunc.PythonModel):

    def load_context(self, context):
        # No change here. This still loads the YOLO checkpoint.
        _log("[INFO] Loading YOLO checkpoint...")
        self.model = YOLO(context.artifacts["checkpoint"])
        _log("[INFO] YOLO model successfully loaded")

    def predict(self, context, model_input: pd.DataFrame):
        """
        model_input contains a column 'fish' with video file paths.
        The old code also failed silently if a video couldn't be opened.
        Now errors are printed so Model Serving logs show what goes wrong.
        """

        # Defensive prints to confirm schema and count
        _log(f"[INFO] Starting prediction. model_input columns: {list(model_input.columns)}")
        _log(f"[INFO] Number of rows received: {len(model_input)}")

        if "fish" not in model_input.columns:
            _log("[ERROR] Expected a 'fish' column with video paths. No such column found.")
            return pd.DataFrame([])

        all_rows = []

        for idx, video_path in enumerate(model_input["fish"]):
            _log(f"[INFO] Processing row {idx}: {video_path}")

            try:
                video_name = os.path.basename(video_path)

                # Read each frame from the video
                for frame_number, frame in read_video_frames(video_path):

                    if frame_number % 10 == 0:
                        _log(f"[INFO] Running YOLO on frame {frame_number}")

                    result = self.model(frame)

                    # Fix for the error: YOLO can return Results OR [Results].
                    # The old code assumed a single object and crashed.
                    if isinstance(result, list):
                        if len(result) == 0:
                            _log("[INFO] YOLO returned an empty list")
                            dets = []
                        else:
                            dets = convert_yolo_to_dets(result[0])
                    else:
                        dets = convert_yolo_to_dets(result)

                    for det in dets:
                        all_rows.append({
                            "fish_id": None,
                            "video": video_name,
                            "frame": frame_number,
                            **det
                        })

            except Exception as e:
                # Previously errors were silent. Added logging so you see what failed.
                _log(f"[ERROR] Failed while processing {video_path}: {e}")

        _log(f"[INFO] Prediction complete. Total detections: {len(all_rows)}")
        return pd.DataFrame(all_rows)






In [0]:
import pandas as pd
from mlflow.models.signature import ModelSignature
from mlflow.types import Schema, ColSpec

input_example = pd.DataFrame({
    "fish": ["/Volumes/prd_dash_lab/dash_data_science_unrestricted/shared_external_volume/rachels_stuff/4_2021-07-06_14-11-37x.mp4"]
})


signature = ModelSignature(
    inputs=Schema([ColSpec("string", "fish")]),
    outputs=Schema([
        ColSpec("integer", "fish_id"),
        ColSpec("string", "video"),
        ColSpec("integer", "frame"),
        ColSpec("float", "x1"),
        ColSpec("float", "y1"),
        ColSpec("float", "x2"),
        ColSpec("float", "y2"),
        ColSpec("float", "confidence")
    ])
)

In [0]:
with mlflow.start_run() as run:

    run_id = run.info.run_id   # capture BEFORE log_model closes the run

    mlflow.pyfunc.log_model(
        name=None,                      # we won't register here
        artifact_path="model",          # path inside this run’s artifacts
        python_model=FishVideoDetector(),
        input_example=input_example,
        artifacts={"checkpoint": model_path},
        signature=signature
    )

print("Logged model under run:", run_id)

### Register model

In [0]:
mlflow.set_registry_uri("databricks-uc")

In [0]:
result = mlflow.register_model(
    model_uri=f"runs:/{run_id}/model",
    name="prd_dash_lab.dash_data_science_unrestricted.FishVideoDetector"
)

print("Registered model version:", result.version)

### Test registered model ( I didnt test for UC model since endpoints in UC models serving does not work yet. Skip to workspace registry)

In [0]:
model = mlflow.pyfunc.load_model("models:/prd_dash_lab.dash_data_science_unrestricted.salmon_model_test/9")

In [0]:
import pandas as pd
import os

# Directory where images are stored
image_dir = "/Volumes/prd_dash_lab/dash_data_science_unrestricted/shared_external_volume/rachels_stuff/"

# List all files (filter for jpg/png)
image_files = [f for f in os.listdir(image_dir) if f.endswith((".mp4"))]

# Build full paths
full_paths = [os.path.join(image_dir, f) for f in image_files]

# Create a DataFrame suitable for your model
input_df = pd.DataFrame({
    "fish": full_paths
})

print(input_df.head())


In [0]:
results = model.predict(input_df)


In [0]:
display(results)

### Register model to workspace

In [0]:
mlflow.set_registry_uri("databricks")

result = mlflow.register_model(
    model_uri=f"runs:/{run_id}/model",
    name="FishVideoDetector"
)

print("Registered model version:", result.version)


In [0]:
import mlflow.pyfunc
import pandas as pd

mlflow.set_registry_uri("databricks")

model_uri = "models:/FishVideoDetector/2"
model = mlflow.pyfunc.load_model(model_uri)

input_df = pd.DataFrame({
    "fish": [
        "/Volumes/prd_dash_lab/dash_data_science_unrestricted/shared_external_volume/rachels_stuff/4_2021-07-06_14-11-37x.mp4"
    ]
})

preds = model.predict(input_df)
print(preds)


In [0]:
print(preds)

In [0]:
%pip install -r /local_disk0/repl_tmp_data/ReplId-19c33-9d7a9-0/tmp7dl33gm3/requirements.txt